# Phase 2 : GLM, le socle actuariel
## Moteur de tarification IARD, données freMTPL2

**Objectif** : construire le tarif de référence du projet à partir de modèles
linéaires généralisés. Fréquence (Poisson avec offset), coût moyen (Gamma),
recombinaison en prime pure, diagnostics et évaluation hors échantillon.
Ce modèle sert de référence de performance (benchmark GBM, Phase 3) et
d'interprétabilité (comparaison SHAP, Phase 4).

**Données** : `data/processed/fremtpl2_clean.parquet`, produit par le notebook
`01_eda` (678 013 polices, nettoyées et enrichies).

**Architecture décidée en Phase 1** :
- fréquence : GLM Poisson, cible `ClaimNb`, offset $\log(\text{Exposure})$ ;
- coût moyen : GLM Gamma lien log, cible `ClaimAmount / ClaimNbSev`, pondéré par `ClaimNbSev` ;
- prime pure : fréquence × coût moyen × facteur correctif $N^{\text{sev}}/N = 0.732$.

## Journal des décisions (suite)

Suite du journal de la Phase 1 (entrées #1 à #21 dans `01_eda`). Alimenté au fil
de la modélisation.

| # | Constat | Décision | Justification | Impact |
|---|---------|----------|---------------|--------|
| 22 | Chiffres de référence de la Phase 1 calculés avant plafonnements (fréquence 0.1007, PP 167.11, taux 0.732) | Références recalculées sur le jeu nettoyé : fréquence 0.1006, PP 167.18 EUR, taux 0.7334 | Cohérence avec les données réellement modélisées | Facteur correctif calculé dynamiquement, jamais codé en dur |
| 23 | 38 % des polices partagent leur profil complet ; IDpol consécutifs (78.5 % à moins de 100, contre 0 % au hasard) : même unité de risque sur plusieurs lignes | Découpage train/test 80/20 **par profil** (`GroupShuffleSplit`, seed 2026), sauvegardé dans `split_train_test.parquet` | Éviter la fuite d'information d'un même assuré entre train et test, critique pour le GBM | Même test en Phase 2 et Phase 3 : benchmark équitable |
| 24 | Modalités `VehGas` entourées d'apostrophes (artefact OpenML) | Apostrophes supprimées | Lisibilité des relativités et de l'application | Modalités : `Diesel`, `Regular` |
| 25 | Écart train/test : fréquence -1.9 % ($z = -1.40$, $p$ exact $= 0.165$), prime pure -4.6 % ; sinistre de 4.08 M EUR dans le train ; IC bootstrap de la PP test à ±20 % | Découpage conservé, graine inchangée (pas de *data snooping*) | Écart de fréquence non significatif ; écart de PP inversé après écrêtement (+2.7 %) : bruit de queue | Évaluation par composante et par indicateurs de rang ; écrêtement des graves à trancher au GLM sévérité |


## 0. Setup et chargement

Imports, options d'affichage, résolution de la racine du projet (même mécanisme
qu'en Phase 1, via le marqueur `.git`) et chargement du jeu nettoyé.

Avant toute modélisation, on vérifie que le fichier chargé est bien celui validé
en Phase 1 : on doit retrouver les chiffres de référence (fréquence 0.1007,
prime pure 167.11 EUR, 24 944 polices dans l'échantillon sévérité). Un écart
signalerait une régénération involontaire du parquet, et invaliderait toute la
suite.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

# Affichage : toutes les colonnes, 4 decimales (frequences de l'ordre de 0.1)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")


def get_project_root(marker: str = ".git") -> Path:
    """Remonte l'arborescence jusqu'au dossier contenant le marqueur (.git)."""
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Racine du projet introuvable (marqueur : {marker})")


PROJECT_ROOT = get_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
for d in (MODELS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_DIR / "fremtpl2_clean.parquet")
print(f"Dimensions : {df.shape}")
df.head()

Dimensions : (678013, 18)


,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,ClaimAmount,ClaimNbSev,DrivAgeBand,VehAgeBand,log_Density,RegionGrouped
0,1,1,0.1000,D,5,0,55,50,B12,'Regular',1217,R82,0.0000,0,46-60,0-1,7.1041,R82
1,3,1,0.7700,D,5,0,55,50,B12,'Regular',1217,R82,0.0000,0,46-60,0-1,7.1041,R82
2,5,1,0.7500,B,6,2,52,50,B12,'Diesel',54,R22,0.0000,0,46-60,2-5,3.9890,Autres
3,10,1,0.0900,B,7,0,46,50,B12,'Diesel',76,R72,0.0000,0,46-60,0-1,4.3307,R72
4,11,1,0.8400,B,7,0,46,50,B12,'Diesel',76,R72,0.0000,0,46-60,0-1,4.3307,R72


In [2]:
# --- CONTROLE DE COHERENCE AVEC LA PHASE 1 ---
# On doit retrouver exactement les chiffres de reference de l'EDA.
freq_ref = df["ClaimNb"].sum() / df["Exposure"].sum()
pp_ref = df["ClaimAmount"].sum() / df["Exposure"].sum()
n_sev = ((df["ClaimNb"] > 0) & (df["ClaimAmount"] > 0)).sum()
taux_obs = df["ClaimNbSev"].sum() / df["ClaimNb"].sum()

print(f"Frequence de reference : {freq_ref:.4f}   (attendu 0.1007)")
print(f"Prime pure de reference : {pp_ref:.2f}  (attendu 167.11)")
print(f"Echantillon severite    : {n_sev:,}    (attendu 24 944)")
print(f"Taux d'observation      : {taux_obs:.4f}   (attendu 0.732)")

# --- PLAFONNEMENTS DE LA PHASE 1 BIEN APPLIQUES ---
assert df["Exposure"].max() <= 1.0, "Exposition non plafonnee"
assert df["ClaimNb"].max() <= 4, "ClaimNb non ecrete"

# --- TYPES DES VARIABLES TARIFAIRES ---
# Le type conditionne le traitement dans la formule GLM (categoriel ou continu)
# et l'ordre d'affichage des modalites dans les tables de relativites.
variables = ["DrivAgeBand", "VehAgeBand", "VehPower", "VehBrand", "VehGas",
             "RegionGrouped", "log_Density", "BonusMalus"]
print("\nTypes des variables tarifaires :")
print(df[variables].dtypes)

Frequence de reference : 0.1006   (attendu 0.1007)
Prime pure de reference : 167.18  (attendu 167.11)
Echantillon severite    : 24,944    (attendu 24 944)
Taux d'observation      : 0.7334   (attendu 0.732)

Types des variables tarifaires :
DrivAgeBand      category
VehAgeBand       category
VehPower            int64
VehBrand         category
VehGas                str
RegionGrouped         str
log_Density       float64
BonusMalus          int64
dtype: object


**Lecture du contrôle de cohérence.**

Le fichier chargé est bien celui de la Phase 1 (678 013 polices, échantillon
sévérité de 24 944 polices identique). Les légers écarts sur les chiffres de
référence ne sont pas une anomalie : dans `01_eda`, ils ont été calculés
**avant** les plafonnements des sections 6.2 et 6.3, alors que le parquet
contient les données **après** plafonnement.

| Indicateur | Phase 1 (avant plafonnement) | Phase 2 (jeu nettoyé) | Origine de l'écart |
|---|---|---|---|
| Fréquence | 0.1007 | 0.1006 | exposition plafonnée (+0.04 %) et `ClaimNb` écrêté à 4 (quelques dizaines de sinistres retirés) |
| Prime pure | 167.11 EUR | 167.18 EUR | exposition plafonnée : $167.11 \times 1.0004 \approx 167.18$, charge inchangée |
| Taux d'observation $N^{\text{sev}}/N$ | 0.732 | 0.7334 | écrêtement de $N$, $N^{\text{sev}}$ inchangé |

**Décision.** Les références de la Phase 2 sont celles du jeu nettoyé. Le facteur
correctif de recombinaison est **recalculé sur les données** plutôt que codé en
dur : c'est la seule façon de garantir que la prime pure recombinée retombe
exactement sur la charge observée.

**Types.** `VehGas` et `RegionGrouped` sont stockées en chaînes (`str`, type
chaîne de pandas 3). On harmonise toutes les variables tarifaires catégorielles
en type `category`, pour garantir leur traitement correct dans les formules GLM
et un ordre d'affichage stable des modalités.

In [3]:
# --- REFERENCES DE LA PHASE 2 (jeu nettoye) ---
# Recalculees sur les donnees, jamais codees en dur : elles servent de
# cibles de controle a la recombinaison en prime pure.
FREQ_REF = df["ClaimNb"].sum() / df["Exposure"].sum()
PP_REF = df["ClaimAmount"].sum() / df["Exposure"].sum()
TAUX_OBS = df["ClaimNbSev"].sum() / df["ClaimNb"].sum()

print(f"FREQ_REF = {FREQ_REF:.4f}")
print(f"PP_REF   = {PP_REF:.2f} EUR")
print(f"TAUX_OBS = {TAUX_OBS:.4f}")

# --- HARMONISATION DES TYPES CATEGORIELS ---
# Le type 'category' garantit un traitement categoriel dans les formules
# statsmodels (patsy) et un ordre de modalites stable.
# Les bandes d'age (issues de pd.cut) sont deja ordonnees : on n'y touche pas.
for col in ["VehBrand", "VehGas", "RegionGrouped"]:
    df[col] = df[col].astype("category")

print(f"\npandas {pd.__version__}")
print(df[["DrivAgeBand", "VehAgeBand", "VehBrand", "VehGas", "RegionGrouped"]].dtypes)

FREQ_REF = 0.1006
PP_REF   = 167.18 EUR
TAUX_OBS = 0.7334

pandas 3.0.3
DrivAgeBand      category
VehAgeBand       category
VehBrand         category
VehGas           category
RegionGrouped    category
dtype: object


## 1. Découpage train/test

### 1.1 Enjeu : évaluer sur des assurés jamais vus

Un modèle de tarification sert à tarifer de **nouveaux** assurés. Sa performance
doit donc être mesurée sur des polices absentes de l'apprentissage : c'est le rôle
de l'échantillon de test (20 % du portefeuille).

Ce découpage conditionne aussi la Phase 3 : le benchmark GLM vs GBM n'a de valeur
que si les deux modèles sont évalués **exactement sur les mêmes polices**. Le
découpage est donc construit une seule fois, sauvegardé, puis réutilisé.

### 1.2 Le risque : un même assuré des deux côtés

freMTPL2 ne contient pas d'identifiant d'assuré, seulement un identifiant de
police (`IDpol`). Or un même assuré peut détenir plusieurs polices (plusieurs
contrats, ou un même contrat découpé en périodes). Ces polices partagent alors
**exactement** les mêmes caractéristiques : âge, bonus-malus, véhicule, commune.

Avec un tirage aléatoire ligne à ligne, un même assuré peut se retrouver à la fois
dans le train et dans le test. Le test ne mesure alors plus seulement la capacité
à tarifer un nouvel assuré, mais aussi celle à **reconnaître un assuré déjà vu**.
C'est une **fuite d'information** :

- faible pour un GLM, dont la structure additive ne peut pas mémoriser un individu ;
- réelle pour un GBM (Phase 3), capable d'isoler des profils très précis et d'en
  apprendre la sinistralité propre, ce qui gonflerait artificiellement ses
  performances.

On mesure d'abord l'ampleur du phénomène : combien de polices partagent leur
profil complet de covariables avec au moins une autre ?

In [4]:
# --- PROFILS DE COVARIABLES ---
# Un "profil" = combinaison exacte des covariables brutes.
# Deux polices au meme profil sont vraisemblablement le meme assure.
# On utilise les variables BRUTES (et non les bandes) : deux assures differents
# peuvent partager une bande d'age, beaucoup moins un age exact + une commune
# (Density) + un bonus-malus + un vehicule identiques.
GROUP_COLS = ["Area", "VehPower", "VehAge", "DrivAge", "BonusMalus",
              "VehBrand", "VehGas", "Density", "Region"] # variables brutes

df["profil_id"] = df.groupby(GROUP_COLS, observed=True, sort=False).ngroup() # profil unique pour chaque combinaison de covariables
df["taille_profil"] = df.groupby("profil_id")["IDpol"].transform("size") # nombre de polices partageant le meme profil

n_polices = len(df) # nombre total de polices
n_profils = df["profil_id"].nunique() # nombre de profils distincts
part_partagees = (df["taille_profil"] > 1).mean() # proportion de polices partagees (taille_profil > 1)

print(f"Polices                             : {n_polices:,}") 
print(f"Profils distincts                   : {n_profils:,}")
print(f"Polices par profil (moyenne)        : {n_polices / n_profils:.2f}")
print(f"Part des polices a profil partage   : {part_partagees:.1%}")

# --- DISTRIBUTION DE LA TAILLE DES PROFILS ---
tailles = pd.cut(df["taille_profil"], bins=[0, 1, 2, 5, 10, 50, np.inf],
                 labels=["1", "2", "3-5", "6-10", "11-50", "50+"])
print("\nRepartition des polices selon la taille de leur profil :")
print(tailles.value_counts(normalize=True).sort_index())

# --- SIGNATURE D'UN MEME ASSURE ---
# Si les polices d'un profil sont des periodes successives d'un meme contrat,
# leur exposition cumulee depasse souvent 1 an, ce qui est impossible
# pour une police unique mais naturel pour un assure suivi plusieurs annees.
expo_profil = df[df["taille_profil"] > 1].groupby("profil_id")["Exposure"].sum()
print(f"\nProfils partages dont l'exposition cumulee depasse 1 an : "
      f"{(expo_profil > 1).mean():.1%}")

# --- PROFIL LE PLUS DUPLIQUE ---
top = df.loc[df["taille_profil"].idxmax()]
print(f"\nProfil le plus duplique ({top['taille_profil']} polices) :")
print(top[GROUP_COLS])

Polices                             : 678,013
Profils distincts                   : 528,765
Polices par profil (moyenne)        : 1.28
Part des polices a profil partage   : 38.0%

Repartition des polices selon la taille de leur profil :
taille_profil
1       0.6196
2       0.2585
3-5     0.0863
6-10    0.0296
11-50   0.0060
50+     0.0000
Name: proportion, dtype: float64

Profils partages dont l'exposition cumulee depasse 1 an : 5.8%

Profil le plus duplique (22 polices) :
Area                  B
VehPower              9
VehAge                3
DrivAge              35
BonusMalus           64
VehBrand             B2
VehGas        'Regular'
Density              92
Region              R72
Name: 352235, dtype: object


In [5]:
# --- VERIFICATION : CONTRAT UNIQUE DECOUPE EN PLUSIEURS LIGNES ? ---
partages = df[df["taille_profil"] > 1] # polices partagees

# 1. Exposition cumulee par profil partage
# Hypothese "contrat annuel decoupe" : cumul concentre sous 1 an.
expo_cumul = partages.groupby("profil_id")["Exposure"].sum() # cumul par profil partage
print("Exposition cumulee des profils partages :")
print(expo_cumul.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]))

# 2. Ecart d'IDpol au sein d'un profil, rapporte a la taille du profil
# Des lignes d'un meme contrat saisies ensemble ont des IDpol consecutifs :
# l'etendue (max - min) est alors proche de (taille - 1).
etendue = partages.groupby("profil_id")["IDpol"].agg(lambda s: s.max() - s.min()) # etendue d'IDpol par profil partage 
taille = partages.groupby("profil_id").size() # nombre de polices par profil partage
consecutifs = (etendue == taille - 1) # profils a IDpol consecutifs
print(f"\nProfils partages a IDpol strictement consecutifs : {consecutifs.mean():.1%}")
print(f"Profils partages a etendue d'IDpol < 100         : {(etendue < 100).mean():.1%}")

# 3. Comparaison avec des paires de polices tirees au hasard (reference)
rng = np.random.default_rng(2026) # pour reproductibilite
paires = rng.choice(df["IDpol"].to_numpy(), size=(10_000, 2)) # paires aleatoires d'IDpol
ecart_hasard = np.abs(paires[:, 0] - paires[:, 1]) # ecart d'IDpol pour les paires aleatoires
print(f"Paires aleatoires a ecart d'IDpol < 100          : {(ecart_hasard < 100).mean():.1%}")

# 4. Exemple : le profil le plus duplique
top_id = df.loc[df["taille_profil"].idxmax(), "profil_id"] # profil_id du profil le plus duplique
print("\nDetail du profil le plus duplique :")
print(df.loc[df["profil_id"] == top_id,
             ["IDpol", "Exposure", "ClaimNb", "ClaimAmount"]].sort_values("IDpol"))

# --- NETTOYAGE DES APOSTROPHES (artefact OpenML) ---
print(f"\nModalites VehGas avant : {list(df['VehGas'].cat.categories)}")
df["VehGas"] = df["VehGas"].cat.rename_categories(lambda c: c.strip("'"))
print(f"Modalites VehGas apres : {list(df['VehGas'].cat.categories)}")

Exposition cumulee des profils partages :
count   108,663.0000
mean          0.7778
std           0.3733
min           0.0055
25%           0.5282
50%           0.8500
75%           0.9900
90%           0.9900
95%           1.1400
max          14.5200
Name: Exposure, dtype: float64

Profils partages a IDpol strictement consecutifs : 45.6%
Profils partages a etendue d'IDpol < 100         : 78.5%
Paires aleatoires a ecart d'IDpol < 100          : 0.0%

Detail du profil le plus duplique :
          IDpol  Exposure  ClaimNb  ClaimAmount
352235  2285383    0.6600        0       0.0000
352236  2285384    0.6600        0       0.0000
352238  2285386    0.6600        0       0.0000
352239  2285387    0.6600        0       0.0000
352240  2285388    0.6600        0       0.0000
352241  2285389    0.6600        0       0.0000
352242  2285390    0.6600        0       0.0000
352243  2285391    0.6600        0       0.0000
352244  2285392    0.6600        0       0.0000
352245  2285393    0.6600    

**Lecture : les profils partagés sont une même unité de risque.**

*Ampleur.* 38 % des polices partagent leur profil complet (âge exact,
bonus-malus, véhicule, commune) avec au moins une autre police. Les 678 013
polices ne représentent que 528 765 profils distincts.

*Origine commune prouvée.* 78.5 % des profils partagés ont des `IDpol` distants
de moins de 100 (45.6 % strictement consécutifs), contre 0 % pour des paires de
polices tirées au hasard. Ces lignes ont été saisies ensemble : ce ne sont pas des
coïncidences de profil, mais une même source.

*Deux mécanismes.*
- **Contrat découpé en périodes** : pour la majorité des profils partagés,
  l'exposition cumulée reste sous 1 an (médiane 0.85, 3e quartile 0.99), signature
  d'un contrat annuel scindé en plusieurs lignes (avenants, changements de
  situation). Cohérent avec l'exposition moyenne très basse du portefeuille (0.53).
- **Lignes répliquées** : le profil le plus dupliqué compte 22 lignes consécutives
  de même exposition (0.66), soit 14.52 années cumulées. Ce n'est pas un découpage
  temporel mais une réplication (flotte de véhicules identiques ou artefact de
  construction du jeu).

*Conséquence.* Quel que soit le mécanisme, ces lignes ne sont pas des observations
indépendantes. Un découpage ligne à ligne en placerait une partie dans le train et
l'autre dans le test : fuite d'information. **Le découpage se fera par profil.**

*Artefact corrigé.* Les modalités de `VehGas` contenaient des apostrophes
(`'Diesel'`, `'Regular'`), héritées de l'import OpenML. Nettoyées pour des tables
de relativités et une application lisibles.

### 1.3 Construction du découpage groupé

On tire 20 % des **profils** (et non des polices) pour constituer le test, avec
`GroupShuffleSplit` de scikit-learn. Toutes les polices d'un même profil tombent
ainsi du même côté.

Trois contrôles valident le découpage :
1. **étanchéité** : aucun profil présent dans les deux échantillons ;
2. **représentativité** : part du test proche de 20 % (le tirage porte sur des
   profils de tailles inégales, d'où un léger écart possible), fréquences et primes
   pures proches des références ;
3. **fuite évitée** : on mesure, par comparaison, la part du test qui aurait eu un
   jumeau dans le train avec un découpage ligne à ligne.

Le découpage est sauvegardé (`split_train_test.parquet`, clé `IDpol`) : la Phase 3
le relira à l'identique. La graine est fixée pour la reproductibilité.

In [6]:
from sklearn.model_selection import GroupShuffleSplit

SPLIT_PATH = DATA_DIR / "split_train_test.parquet"
SEED = 2026

# --- CREATION OU RELECTURE DU DECOUPAGE ---
# Cree une seule fois, puis relu : garantit le meme test en Phase 2 et Phase 3.
if SPLIT_PATH.exists():
    split = pd.read_parquet(SPLIT_PATH).set_index("IDpol")["split"]
    df["split"] = df["IDpol"].map(split)
    print("Decoupage existant relu.")
else:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    _, test_idx = next(gss.split(df, groups=df["profil_id"]))
    df["split"] = "train"
    df.loc[df.index[test_idx], "split"] = "test"
    df[["IDpol", "split"]].to_parquet(SPLIT_PATH, index=False)
    print("Decoupage cree et sauvegarde.")

assert df["split"].notna().all(), "Polices sans affectation train/test"

# --- CONTROLE 1 : ETANCHEITE ---
profils_mixtes = (df.groupby("profil_id")["split"].nunique() > 1).sum()
print(f"\nProfils presents des deux cotes : {profils_mixtes}  (attendu 0)")

# --- CONTROLE 2 : REPRESENTATIVITE ---
bilan = df.groupby("split").agg(
    polices=("IDpol", "size"),
    profils=("profil_id", "nunique"),
    exposition=("Exposure", "sum"),
    sinistres=("ClaimNb", "sum"),
    charge=("ClaimAmount", "sum"),
)
bilan["part_polices"] = bilan["polices"] / bilan["polices"].sum()
bilan["frequence"] = bilan["sinistres"] / bilan["exposition"]
bilan["prime_pure"] = bilan["charge"] / bilan["exposition"]
print("\nBilan du decoupage :")
print(bilan)
print(f"\nReferences : frequence {FREQ_REF:.4f} | prime pure {PP_REF:.2f} EUR")

# --- CONTROLE 3 : FUITE EVITEE ---
# Simulation d'un decoupage ligne a ligne : part des polices test dont le
# profil figure aussi dans le train.
rng = np.random.default_rng(SEED)
test_alea = rng.random(len(df)) < 0.2
profils_train_alea = set(df.loc[~test_alea, "profil_id"])
fuite_alea = df.loc[test_alea, "profil_id"].isin(profils_train_alea).mean()
print(f"\nDecoupage ligne a ligne : {fuite_alea:.1%} du test aurait un jumeau dans le train")
print("Decoupage groupe        : 0.0 % par construction")

# --- ECHANTILLONS DE TRAVAIL ---
train = df[df["split"] == "train"].copy()
test = df[df["split"] == "test"].copy()
print(f"\ntrain : {len(train):,} polices | test : {len(test):,} polices")

Decoupage cree et sauvegarde.

Profils presents des deux cotes : 0  (attendu 0)

Bilan du decoupage :
       polices  profils   exposition  sinistres          charge  part_polices  \
split                                                                           
test    135616   105753  71,579.5329       7096 11,525,316.8800        0.2000   
train   542397   423012 286,780.5725      28960 48,383,899.6200        0.8000   

       frequence  prime_pure  
split                         
test      0.0991    161.0141  
train     0.1010    168.7140  

References : frequence 0.1006 | prime pure 167.18 EUR

Decoupage ligne a ligne : 32.5% du test aurait un jumeau dans le train
Decoupage groupe        : 0.0 % par construction

train : 542,397 polices | test : 135,616 polices


### 1.4 L'écart train/test est-il du bruit ?

Le découpage est étanche et évite une fuite de 32.5 %. Reste un déséquilibre
apparent entre les deux échantillons :

| Indicateur | Train | Test | Écart relatif |
|---|---|---|---|
| Fréquence | 0.1010 | 0.0991 | -1.9 % |
| Prime pure | 168.71 EUR | 161.01 EUR | -4.6 % |

Avant de conclure à un découpage biaisé, on vérifie si ces écarts relèvent de la
simple fluctuation d'échantillonnage.

**Fréquence.**

*Notations.* Pour une police $i$ :
- $N_i$ : nombre de sinistres de la police, **variable aléatoire** ;
- $e_i$ : exposition de la police (fraction d'année couverte), **constante connue** ;
- $\lambda_i$ : fréquence annuelle de la police, paramètre inconnu.

Hypothèse de travail : $N_i \sim \mathcal{P}(\lambda_i \, e_i)$, les $N_i$ étant
indépendants entre polices (c'est l'indépendance que le découpage par profil
cherche justement à préserver entre train et test).

Pour un échantillon $S$ (train ou test), on note :
- $N_S = \sum_{i \in S} N_i$ : nombre total de sinistres, **variable aléatoire** ;
- $E_S = \sum_{i \in S} e_i$ : exposition totale, **constante connue** ;
- $\hat{\lambda}_S = N_S / E_S$ : fréquence estimée de l'échantillon.

*Variance de la fréquence estimée.* Une somme de Poissons indépendantes est une
Poisson, donc $N_S \sim \mathcal{P}\big(\sum_{i \in S} \lambda_i e_i\big)$, et
$\mathbb{V}(N_S) = \mathbb{E}(N_S)$. Comme $E_S$ est une constante :

$$
\mathbb{V}(\hat{\lambda}_S) = \frac{\mathbb{V}(N_S)}{E_S^2} = \frac{\mathbb{E}(N_S)}{E_S^2},
\qquad \text{estimée par } \frac{N_S}{E_S^2}
$$

en remplaçant l'espérance inconnue par sa valeur observée. Ce résultat ne suppose
**pas** que toutes les polices ont la même fréquence.

*Statistique de test.* On teste $H_0$ : les deux échantillons ont la même
fréquence moyenne. Train et test étant disjoints, $\hat{\lambda}_{\text{train}}$
et $\hat{\lambda}_{\text{test}}$ sont indépendants : la variance de leur écart
est la somme de leurs variances. Avec plusieurs milliers de sinistres de chaque
côté, le TCL rend chaque fréquence estimée approximativement gaussienne, d'où la
statistique de Wald :

$$
z = \frac{\hat{\lambda}_{\text{test}} - \hat{\lambda}_{\text{train}}}
{\sqrt{\dfrac{N_{\text{train}}}{E_{\text{train}}^2} + \dfrac{N_{\text{test}}}{E_{\text{test}}^2}}}
\ \underset{H_0}{\approx}\ \mathcal{N}(0, 1)
$$

L'écart est non significatif au seuil de 5 % si $|z| < 1.96$.

*Contrôle exact complémentaire.* Sous $H_0$ avec une fréquence commune, et
conditionnellement au nombre total de sinistres $N_{\text{tot}} = N_{\text{train}} + N_{\text{test}}$,
chaque sinistre tombe dans le test avec une probabilité égale à la part
d'exposition du test :

$$
N_{\text{test}} \mid N_{\text{tot}} \sim \mathcal{B}\!\left(N_{\text{tot}},\ \frac{E_{\text{test}}}{E_{\text{train}} + E_{\text{test}}}\right)
$$

(propriété classique : des Poissons indépendantes conditionnées à leur somme
suivent une loi multinomiale). Ce test ne repose sur aucune approximation normale.

*Robustesse.* En cas de surdispersion, $\mathbb{V}(N_S) > \mathbb{E}(N_S)$ :
la vraie variance dépasse $N_S / E_S^2$ et le vrai $|z|$ est plus faible. Une
conclusion de non-significativité obtenue sous Poisson reste donc valable.

**Prime pure.** Elle est dominée par la queue de distribution (en Phase 1, le
top 1 % des sinistres portait 37 % de la charge, avec un maximum à 4 M EUR). Sur
une charge de test de 11.5 M EUR, un seul sinistre grave suffit à déplacer la
prime pure de plusieurs pourcents. Trois vérifications :
1. la répartition des plus gros sinistres entre train et test ;
2. la prime pure **écrêtée** au 99e percentile : si l'écart disparaît, il vient
   des sinistres graves et non d'un déséquilibre de profils ;
3. un intervalle de confiance **bootstrap** de la prime pure du test, obtenu en
   rééchantillonnant les **profils** (et non les polices), pour respecter la
   dépendance mise en évidence en section 1.2.

**Règle de méthode.** Quel que soit le résultat, la graine n'est pas modifiée.
Tester plusieurs graines jusqu'à obtenir un découpage « équilibré » reviendrait
à choisir son échantillon de test en fonction de ses résultats (*data snooping*).
Un écart expliqué par le bruit est documenté, pas corrigé.

In [7]:
from scipy.stats import binomtest

# --- 1. TEST DE L'ECART DE FREQUENCE ---
# N_S : nombre total de sinistres (aleatoire), E_S : exposition totale (constante).
n_tr, e_tr = bilan.loc["train", ["sinistres", "exposition"]]
n_te, e_te = bilan.loc["test", ["sinistres", "exposition"]]

# 1a. Test de Wald : ecart des frequences / ecart-type estime sous Poisson
diff = n_te / e_te - n_tr / e_tr
se = np.sqrt(n_tr / e_tr**2 + n_te / e_te**2)
z = diff / se
print(f"Ecart de frequence test - train : {diff:+.4f}")
print(f"Statistique z (Wald)            : {z:+.2f}  (|z| < 1.96 : non significatif a 5 %)")

# 1b. Test exact : binomiale conditionnelle au nombre total de sinistres
p0 = e_te / (e_tr + e_te)                       # part d'exposition du test
n_tot = int(n_tr + n_te)
res = binomtest(int(n_te), n_tot, p0)
print(f"Sinistres test attendus sous H0 : {p0 * n_tot:,.0f} (observes : {n_te:,.0f})")
print(f"p-value binomiale exacte        : {res.pvalue:.3f}")

# --- 2. OU SONT TOMBES LES GROS SINISTRES ? ---
print("\n10 plus grosses charges par police :")
print(df.nlargest(10, "ClaimAmount")[["IDpol", "ClaimAmount", "split"]])

# --- 3. PRIME PURE ECRETEE ---
# Charge par police ecretee au 99e percentile des charges positives.
# Si l'ecart train/test disparait, il vient des sinistres graves (bruit de queue).
seuil = df.loc[df["ClaimAmount"] > 0, "ClaimAmount"].quantile(0.99)
charge_ecr = df["ClaimAmount"].clip(upper=seuil)
pp_ecr = (
    charge_ecr.groupby(df["split"]).sum()
    / df.groupby("split")["Exposure"].sum()
)
print(f"\nSeuil d'ecretement (99e percentile) : {seuil:,.0f} EUR")
print("Prime pure ecretee par echantillon :")
print(pp_ecr)
print(f"Ecart relatif test / train : {pp_ecr['test'] / pp_ecr['train'] - 1:+.1%}")

# --- 4. INCERTITUDE DE LA PRIME PURE DU TEST (bootstrap par profil) ---
# On reechantillonne les PROFILS (et non les polices) pour respecter
# la dependance mise en evidence en section 1.2.
agg_test = test.groupby("profil_id")[["ClaimAmount", "Exposure"]].sum()
charge_p = agg_test["ClaimAmount"].to_numpy()
expo_p = agg_test["Exposure"].to_numpy()
n_prof = len(agg_test)

rng = np.random.default_rng(SEED)
B = 500
pp_boot = np.empty(B)
for b in range(B):
    idx = rng.integers(0, n_prof, n_prof)
    pp_boot[b] = charge_p[idx].sum() / expo_p[idx].sum()

ic_bas, ic_haut = np.percentile(pp_boot, [2.5, 97.5])
pp_train = bilan.loc["train", "prime_pure"]
print(f"\nPrime pure test  : {bilan.loc['test', 'prime_pure']:.2f} EUR")
print(f"IC bootstrap 95 % : [{ic_bas:.2f} ; {ic_haut:.2f}] EUR")
print(f"Prime pure train ({pp_train:.2f}) dans l'IC     : {ic_bas <= pp_train <= ic_haut}")
print(f"Reference globale ({PP_REF:.2f}) dans l'IC    : {ic_bas <= PP_REF <= ic_haut}")

Ecart de frequence test - train : -0.0018
Statistique z (Wald)            : -1.40  (|z| < 1.96 : non significatif a 5 %)
Sinistres test attendus sous H0 : 7,202 (observes : 7,096)
p-value binomiale exacte        : 0.165

10 plus grosses charges par police :
          IDpol    ClaimAmount  split
150027  1120377 4,075,400.5600  train
54317    110846 1,404,185.5200   test
270621  2141337 1,301,172.6000  train
416629  3122016   774,411.5000  train
203399  2008127   399,213.6600  train
368669  3025890   382,955.1400  train
147894  1117644   307,096.4200  train
75668    158309   301,635.4900  train
396360  3075820   287,423.0000  train
431421  3150210   281,897.4900  train

Seuil d'ecretement (99e percentile) : 18,278 EUR
Prime pure ecretee par echantillon :
split
test    119.0856
train   115.9579
dtype: float64
Ecart relatif test / train : +2.7%

Prime pure test  : 161.01 EUR
IC bootstrap 95 % : [130.33 ; 204.56] EUR
Prime pure train (168.71) dans l'IC     : True
Reference globale (167.18) 

**Lecture : le découpage est représentatif, les écarts relèvent du bruit.**

*Fréquence.* Test de Wald : $z = -1.40$ ; test binomial exact : $p = 0.165$.
Le test compte 7 096 sinistres pour 7 202 attendus sous $H_0$. L'écart de
fréquence (0.0991 contre 0.1010) est compatible avec la fluctuation
d'échantillonnage : non significatif au seuil de 5 %.

*Prime pure : un seul sinistre explique l'écart.* La plus grosse charge du
portefeuille (4.08 M EUR) est tombée dans le train. Elle contribue à elle seule
pour $4\,075\,400 / 286\,781 \approx 14.2$ EUR à la prime pure du train, soit près
de deux fois l'écart observé (7.7 EUR). Le test contient lui aussi un sinistre
grave (1.40 M EUR, soit 19.6 EUR de prime pure). Après écrêtement de la charge par
police au 99e percentile (18 278 EUR, calculé ici sur la charge par police et non
sur le coût moyen de la Phase 1), l'écart **s'inverse** : le test dépasse le train
de 2.7 %. Il n'y a donc pas de déséquilibre de profils, seulement du bruit de queue.

*Incertitude de la prime pure observée.* L'intervalle bootstrap à 95 % de la prime
pure du test (rééchantillonnage par profil) est $[130.3\,;\,204.6]$ EUR, soit
environ $\pm 20\,\%$, et il est asymétrique à droite (queue épaisse). La prime pure
du train (168.71) et la référence globale (167.18) y figurent.

**Décision.** Découpage conservé, graine inchangée.

**Conséquences pour la suite.**
- La prime pure observée sur le test est trop instable pour juger un modèle sur son
  seul niveau. L'évaluation portera sur la fréquence et la sévérité séparément,
  puis sur la prime pure au moyen d'indicateurs de classement (courbe de lift,
  indice de Gini), robustes à quelques sinistres extrêmes.
- Le sinistre de 4.08 M EUR, dans le train, influencera le GLM sévérité. La
  question de l'écrêtement des sinistres graves est reportée à la section sévérité.